# 02a – Làm sạch nhẹ và ẩn danh hóa bộ ER (Bronze → Silver)

**Chủ sở hữu: Người 1.** Mục đích: đọc `data/raw/Hospital_Emergency_Room_Data_Raw.csv`, ghi các file `*_er*` trong `data/anonymized/`. Minh chứng cho mục 3.3.1 và 3.5 của báo cáo.

**Nguyên tắc**
- Notebook này là nơi duy nhất ghi các file ER trong `data/anonymized/`. Phần Synthea thuộc notebook `02b` (Thành viên 5).
- Muối, hàm băm, ngưỡng k và cách chia nhóm tuổi nằm trong `src/anon_common.py` (Người 1 sở hữu), dùng chung với `02b` và `02c`.
- Muối lấy từ biến môi trường `ANON_SALT` (hoặc file `.env`), không ghi vào code hay log. Người chạy ghi bằng mã thành viên qua biến `RUN_BY` (ví dụ `TV1`).
- Làm sạch không xóa dòng, chỉ gắn cờ.

Các bước: (1) kiểm tra muối → (2) bảng kê trường định danh → (3) ẩn danh ER → (4) kiểm tra k-anonymity → (5) quét sót định danh → (6) nhật ký chạy.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                            if (p / 'src' / 'anon_common.py').exists()) / 'src'))
import numpy as np
import pandas as pd
from anon_common import (ROOT, RAW, ANON, K_MIN, REF_DATE, AGE_BINS, AGE_LABELS,
                         get_salt, pseudonymize, Auditor, class_sizes, k_report, scan_leftovers)

SALT = get_salt()                      # dừng nếu thiếu, còn giá trị mẫu hoặc ngắn hơn 16 ký tự
audit = Auditor('02a_clean_anonymize_er')

## 2. Bảng kê trường định danh (bộ ER)

In [ ]:
inventory_er = pd.DataFrame([
    ('ER', 'Patient Id', 'Trực tiếp', 'Băm HMAC-SHA256 có muối → patient_hash', 'Có dạng số an sinh xã hội (100% dòng khớp ###-##-####)'),
    ('ER', 'Patient First Inital', 'Trực tiếp', 'Bỏ', 'Một phần tên (tên cột giữ đúng chính tả trong file nguồn, "Inital")'),
    ('ER', 'Patient Last Name', 'Trực tiếp', 'Bỏ', 'Họ tên'),
    ('ER', 'Patient Admission Date', 'Gián tiếp', 'Làm tròn xuống giờ → visit_datetime', 'Ngày giờ chính xác là mốc dễ liên kết; vẫn cần giờ cho heatmap'),
    ('ER', 'Patient Age', 'Gián tiếp', 'Nhóm tuổi 4 bậc → age_group', 'Tuổi chính xác kết hợp giới tính, chủng tộc làm tăng nguy cơ'),
    ('ER', 'Patient Gender', 'Gián tiếp', 'Chuẩn hóa M/F → gender', 'Nhãn không thống nhất (M, F, Male, Female)'),
    ('ER', 'Patient Race', 'Gián tiếp', 'Giữ → race', 'Cần cho phân tích công bằng; k được kiểm tra ở mục 4'),
    ('ER', 'Department Referral / Admission Flag / Satisfaction / Waittime', 'Không định danh', 'Giữ', 'Chỉ số vận hành'),
    ('ER', 'Patient Admission Flag.1', 'Trùng lặp', 'Bỏ', 'Cột lặp, giống hệt Patient Admission Flag'),
], columns=['source', 'field', 'category', 'treatment', 'reason'])
inventory_er.to_csv(ANON / 'identifier_inventory_er.csv', index=False)
inventory_er

## 3. ER → `er_visits_anon.csv`

Lưu ý: nhãn `None` ở `Department Referral` nghĩa là **không chuyển khoa**. Pandas mặc định coi "None" là thiếu dữ liệu nên phải đọc với `keep_default_na=False` rồi chỉ coi chuỗi rỗng là thiếu.

In [ ]:
er = audit.read_csv('Hospital_Emergency_Room_Data_Raw.csv', keep_default_na=False, na_values=[''])
er.columns = [c.strip() for c in er.columns]

# Cột lặp: pandas tự đổi tên cột trùng thành 'Patient Admission Flag.1'
assert (er['Patient Admission Flag'] == er['Patient Admission Flag.1']).all()

dt = pd.to_datetime(er['Patient Admission Date'], format='%d-%m-%Y %H:%M')
gender = er['Patient Gender'].str.strip().str.upper().str[0]
assert gender.isin(['M', 'F']).all()

er_anon = pd.DataFrame({
    'patient_hash': er['Patient Id'].map(pseudonymize),
    'visit_datetime': dt.dt.floor('h').dt.strftime('%Y-%m-%d %H:%M'),
    'gender': gender,
    'age_group': pd.cut(er['Patient Age'], AGE_BINS, right=False, labels=AGE_LABELS).astype(str),
    'race': er['Patient Race'].str.strip(),
    'referral_department': er['Department Referral'].replace({'None': 'No referral'}),
    'is_referred': er['Department Referral'].ne('None'),
    'is_admitted': er['Patient Admission Flag'].astype(bool),
    'satisfaction_score': er['Patient Satisfaction Score'],       # giữ NaN, không điền (thiếu ~73%)
    'wait_time_min': er['Patient Waittime'],
})
assert er_anon['patient_hash'].is_unique                  # băm không va chạm
assert er_anon['age_group'].ne('nan').all()
audit.record['dropped_columns']['er'] = ['Patient Id', 'Patient First Inital', 'Patient Last Name', 'Patient Age',
                                         'Patient Admission Date', 'Patient Admission Flag.1']
audit.write_csv(er_anon, 'er_visits_anon.csv')
er_anon.head()

## 4. Kiểm tra k-anonymity (ER)

k = kích thước nhóm tương đương nhỏ nhất theo bộ thuộc tính gián tiếp. Ràng buộc của nhóm: k ≥ 5 với bộ nhân khẩu học (A). Các bộ có thêm thời gian hoặc khoa (B, C, D) được đo và công bố nhưng không ràng buộc, vì phân tích vận hành cần giờ vào.

In [ ]:
er_k = er_anon.assign(visit_month=er_anon['visit_datetime'].str[:7], visit_hour=er_anon['visit_datetime'].str[11:13])
rows = [
    k_report(er_k, ['age_group', 'gender', 'race'], 'ER', 'A: nhân khẩu học (ràng buộc)'),
    k_report(er_k, ['age_group', 'gender', 'race', 'referral_department'], 'ER', 'B: A + khoa chuyển tuyến'),
    k_report(er_k, ['age_group', 'gender', 'race', 'visit_month'], 'ER', 'C: A + tháng khám'),
    k_report(er_k, ['age_group', 'gender', 'race', 'visit_hour'], 'ER', 'D: A + giờ khám'),
]
k_table = pd.DataFrame(rows)
k_table.to_csv(ANON / 'k_anonymity_report_er.csv', index=False)
audit.record['k_anonymity'] = rows
assert k_table.loc[k_table['qi_set'].str.startswith('A'), 'k_min'].min() >= K_MIN, 'k-anonymity chưa đạt'
k_table.drop(columns='qi_columns')

## 5. Quét sót định danh trong đầu ra ER

In [ ]:
scan_leftovers({'er_visits_anon.csv': er_anon}, raw_ids=set(er['Patient Id']),
               forbidden_cols={'Patient Id', 'Patient Last Name', 'Patient First Inital'})
print('Quét sót: không còn mẫu SSN, mã gốc hay cột định danh trong đầu ra ER.')

## 6. Nhật ký chạy

Ghi vào `anonymization_audit_er.jsonl` (file riêng của notebook này): thời điểm UTC, mã người chạy (`RUN_BY`), SHA-256 và số dòng của file vào/ra, cột đã bỏ, kết quả k-anonymity. Không chứa muối hay dữ liệu bệnh nhân. Nhật ký này chỉ ghi các lần chạy notebook; nhật ký ai mở file hay truy vấn kho do hệ điều hành, kho lưu trữ hoặc hệ quản trị CSDL ghi.

In [ ]:
audit.save('anonymization_audit_er.jsonl')
print('Đã ghi:', *sorted(p.name for p in ANON.glob('*_er*')), sep='\n- ')